In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Import required libraries
import os
import kagglehub
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Write your code here
folder= os.listdir(path)
print(folder)
data= os.listdir(path+'/PlantVillage')
print(data)
train= os.listdir(path+'/PlantVillage/train')
print(train)
pot= os.listdir(path+'/PlantVillage/train/Potato___healthy')
print(pot)

In [ ]:
i = 0
for item in train:
    print(i, item)
    i += 1

In [ ]:
import glob
folder_path = os.path.join(path, "PlantVillage", "train","Potato___healthy", "*.JPG")

from torchvision.datasets import ImageFolder
# Get paths of all .jpg images
image_files = glob.glob(folder_path)
print(image_files[:5])

In [ ]:
### **🔹 General Structure of a Dataset Class**
from torch.utils.data import Dataset
from PIL import Image
import glob

class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir  # Dataset path
        self.transform = transform  # Transformations
        self.class_labels = {}
        self.classes = sorted(os.listdir(root_dir))
        for idx, folder in enumerate(self.classes):
           self.class_labels[folder] = idx


        # Get all image paths
        self.image_paths = []
        self.labels = []
        # train_dataset = ImageFolder(os.path.join(root_dir,class_name), transform=transform)
        for class_name, label in self.class_labels.items():
            class_images = glob.glob(f"{root_dir}/{class_name}/*.JPG")  # Find all images
            self.image_paths.extend(class_images)
            self.labels.extend([label] * len(class_images))  # Assign labels

    def __len__(self):
        return len(self.image_paths)  # Total number of images

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]  # Get image path
        image = Image.open(image_path).convert('RGB')
        label = self.labels[idx]  # Get label

        # Apply transformations (if any)
        if self.transform:
            image = self.transform(image)

        return image, label  # Return processed image & label

In [ ]:
# Create datasets for train, validation, and test
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),  # Rotate images randomly within ±15 degrees
    transforms.ToTensor(),  # Convert to tensor
])

# train_dataset = CustomDataset(
#     root_dir=os.path.join(path, "PlantVillage","train"),
#     transform=transform
# )

# test_dataset = CustomDataset(
#     root_dir=os.path.join(path,"PlantVillage", "test"),
#     transform=transform
# )
train_dataset = ImageFolder(os.path.join(os.path.join(path, "PlantVillage","train")), transform=transform)
test_dataset = ImageFolder(os.path.join(os.path.join(path, "PlantVillage","test")), transform=transform)


print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"Classes: {train_dataset.classes[:3]}...")  # Show classes

In [ ]:
import matplotlib.pyplot as plt
import random

fig, axes = plt.subplots(2, 5)

indices = random.sample(range(len(train_dataset)), 10)

for i, idx in enumerate(indices):
    image, label = train_dataset[idx]
    img = image.permute(1, 2, 0)

    axes.flat[i].imshow(img)
    axes.flat[i].set_title(label) #class_name = train_dataset.classes[label]
    axes.flat[i].axis("off")

plt.show()

In [ ]:
# Write your code hereimport torch.nn as nn
import torch.nn as nn
import torch

# Define the CNN Model
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()

        # Convolutional Layers
        self.conv1 =nn.Sequential(nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),nn.BatchNorm2d(16))
        self.conv2 = nn.Sequential(nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1), nn.BatchNorm2d(16))
        self.conv3 = nn.Sequential(nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),nn.BatchNorm2d(16))
        self.conv4 = nn.Sequential(nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),nn.BatchNorm2d(16))
        self.conv5 = nn.Sequential( nn.Conv2d(in_channels=128, out_channels=1024, kernel_size=3, padding=1),nn.BatchNorm2d(16))


        # Activation
        self.relu = nn.ReLU()

        # Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layers
        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Convolution + ReLU + Pooling
        x = self.pool(self.relu(self.conv1(x)))  # (Batch, 16, 16, 16)
        x = self.pool(self.relu(self.conv2(x)))  # (Batch, 32, 8, 8)
        x = self.pool(self.relu(self.conv3(x)))  # (Batch, 64, 4, 4)
        x = self.pool(self.relu(self.conv4(x)))
        x = self.pool(self.relu(self.conv5(x)))

        # Flatten
        x = x.view(x.size(0), -1)  # (Batch, 64*4*4)

        # Fully Connected Layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Logits

        return x

In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

In [ ]:
# Write your code here
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 4 # Number of epochs

In [ ]:
# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)


# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

    # .BatchNorm2d(16)

In [ ]:
# Write your code here
